# 🎯 AI Product Recommendation Engine — IoT Edition
Recommandations logiques pour une boutique de matériel IoT/électronique.

In [11]:
!pip install pandas numpy scikit-learn sentence-transformers -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import re, unicodedata, warnings
warnings.filterwarnings("ignore")

# Le modèle sémantique (sentence-transformers) est optionnel :
# - en ligne (Colab) -> embeddings sémantiques de haute qualité
# - hors ligne        -> repli automatique sur TF-IDF (aucune installation lourde requise)
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("✅ Dépendances OK — mode sémantique (sentence-transformers)")
except Exception:
    HAS_ST = False
    print("⚠️ sentence-transformers indisponible — repli automatique sur TF-IDF")

✅ Dépendances OK — mode sémantique (sentence-transformers)


## 📊 Étape 1 : Chargement des données

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("cothings-ai/inventory_export.csv"),
    Path("data/inventory_export.csv")
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"✅ {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## 🧹 Étape 2 : Nettoyage + Catégorisation IoT

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    """Texte filtré (sans mots-outils, sans lettres seules) — pour catégories + TF-IDF."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par","ou","en"}
    return " ".join(t for t in text.split() if len(t)>1 and t not in stop)

def norm_raw(text):
    """Texte brut (GARDE les lettres seules : 'type c', 'pile d') — pour l'extraction d'attributs."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+"," ",text).strip()

# ============================================================================
# BRIQUE 0 — CATÉGORISATION COMPLÈTE (objectif : 0 % de "autre")
# ============================================================================
CATEGORY_RULES = {
 "capteur":     ["capteur","capteurs","senseur","sensor","dht11","dht22","ds18b20","hc-sr04","hcsr04","pir","ultrason","mpu6050","mpu9250","bmp","bme280","mq2","mq3","mq135","ldr","acs712","sct013","gy-","lm35","tof","photoresistance","accelerometre","gyroscope","camera","debimetre","yf-s201","flow","ad8232","ecg","hc-sr501","encodeur","mlx","reed","hx711","load cell","fin de course"],
 "carte":       ["raspberry","rpi","arduino","esp32","esp8266","nodemcu","wemos","microbit","stm32","atmega","jetson","wroom","attiny","carte developpement","mega2560","leonardo"],
 "mobilite":    ["voiture","telecommandee","telecommande","drone","quadcopter","helices","helice","roue","roues","chassis","4wd","2wd","robot","bateau","avion","mecanum","suiveur"],
 "moteur":      ["moteur","motor","servo","servomoteur","stepper","nema17","nema23","28byj","pompe","ventilateur","brushless","pas a pas","mg90","mg90s","mg995","mg996","sg90","ds3218","actionneur","vibreur"],
 "led":         ["led","ruban","bande","ampoule","lampe","lumiere","spectre","rgb","neon","matrice","ws2812","cob","horticole","projecteur","spot","luminaire","strip"],
 "batterie":    ["pile","piles","batterie","battery","18650","21700","26650","14500","lipo","li-po","lithium","accu","alcaline","alkaline","nimh","ni-mh","vape","cr2032","cr2016","cr1620","r20","r14"],
 "chargeur":    ["chargeur","charging","chargement","tp4056","imax"],
 "alimentation":["alimentation","tension","regulateur","regulator","convertisseur","transformateur","decoupage","buck","boost","abaisseur","elevateur","pwm","7805","79l12","onduleur","step-down","step-up","fusible"],
 "rf":          ["433mhz","rf","emetteur","recepteur","bluetooth","ble","wifi","gsm","gprs","gps","nrf24","lora","zigbee","sim800","sim900","antenne","hm-10","nfc","rfid"],
 "electronique":["module","circuit","relais","relay","lcd","oled","hdmi","i2c","spi","74hc","mosfet","registre","potentiometre","potentiometer","ecran","interrupteur","bouton","contacteur","rtc","driver","l298","shield","bouclier","pcb","prototype","perforee","controleur","cnc","gravure","expansion","stc-1000","uln2003","afficheur","segments","tm1637","max7219","clavier","membrane","ft232","rs232","cp2102","ch340","keypad","joystick","dip switch","horloge","ne555"],
 "composant":   ["diode","diodes","resistance","resistances","condensateur","transistor","zener","quartz","cristal","inductance","1n4148","1n4007","thyristor","triac","optocoupleur","triode","optotriac"],
 "connectique": ["cable","cables","fil","fils","connecteur","connecteurs","cosses","cosse","jumper","cavalier","dupont","barette","barrette","prise","borne","bornier","header","nappe","gaine","thermoretractable","domino","carte memoire","micro sd","microsd","sd card","cordon","rallonge","serre cable"],
 "mesure":      ["multimetre","testeur","oscilloscope","amperemetrique","pince ampere","compteur","balance","luxmetre","thermometre","wattmetre","pzem","ph metre","electrode","sonde","frequencemetre","generateur de signal"],
 "soudure":     ["souder","soudure","etain","flux","panne","dessoudage","fer a souder","desoudage"],
 "solaire":     ["solaire","photovoltaique","photovolta","mppt","panneau solaire"],
 "audio":       ["haut-parleur","haut parleur","microphone","amplificateur","buzzer","speaker","ampli","sonore","hp","mp3","jack audio"],
 "outillage":   ["tournevis","cle","cles","clef","pince","pinces","scie","cutter","lame","lames","foret","forets","marteau","coffret","outil","outils","douille","douilles","embout","embouts","percage","brucelle","mandrin","cliquet","perceuse","visseuse","meuleuse","burin","lime","etau","cle mixte","jeu de","pistolet","colle","peinture","disque","brosse","meule","nettoyeur","levage","electrogene","laser","niveau","compresseur","ponceuse","rabot","agrafeuse","riveteuse","decapeur","imprimante","creality","filament","truelle","spatule","pinceau","rouleau","echelle","escabeau","cric","palan","aspirateur","disqueuse","sangle"],
 "mecanique":   ["vis","ecrou","ecrous","boulon","rondelle","rondelles","rivet","roulement","engrenage","courroie","poulie","ressort","rail","profile","entretoise","boitier","coque","dissipateur","heatsink","radiateur","colonnette","equerre","charniere","glissiere","tige filetee"],
 "electrique":  ["disjoncteur","contacteur","sectionneur","chint","parafoudre","goulotte","armoire"],
}
# Tokens prouvant qu'un produit n'est PAS une carte (MCU/SBC) mais un accessoire/afficheur/alim
ACCESSORY = {"lcd","oled","ecran","afficheur","ruban","boitier","coque","dissipateur",
             "ventilateur","camera","shield","transformateur","adaptateur","alimentation","chargeur"}
CAT_ORDER = ["chargeur","batterie","led","alimentation","moteur","rf","electronique","composant",
             "connectique","mesure","soudure","solaire","audio","outillage","mecanique","electrique"]

TOOL_BRANDS = {"total","wadfow","harden","yato","tolsen","ingco","stanley","bosch","makita",
               "dewalt","milwaukee","creality","deli"}
def is_tool_brand(title):
    return any(b in norm_raw(title).split() for b in TOOL_BRANDS)

def _has(n, words, kws):
    for k in kws:
        if " " in k:
            if k in n: return True
        elif len(k) <= 3:
            if k in words: return True
        else:
            if any(k in w for w in words) or k in n: return True
    return False

def infer_category(title):
    n = normalize_text(title); words = set(n.split()); wr = set(norm_raw(title).split())
    if _has(n, words, CATEGORY_RULES["mobilite"]): return "mobilite"   # voiture/robot/drone d'abord
    if _has(n, words, CATEGORY_RULES["capteur"]):  return "capteur"
    if _has(n, words, CATEGORY_RULES["carte"]) and not (wr & ACCESSORY): return "carte"
    for cat in CAT_ORDER:
        if _has(n, words, CATEGORY_RULES[cat]): return cat
    if is_tool_brand(title): return "outillage"   # repli : produit de marque outil -> outillage
    return "autre"

# ============================================================================
# SPECS (signal de "qualité" : plus grand = mieux pour mAh / W / canaux)
# ============================================================================
SPEC_PATTERNS = {
 "capacity_mah":(r'\b(\d+(?:[.,]\d+)?)\s*mah\b',50,100000),
 "capacity_ah": (r'\b(\d+(?:[.,]\d+)?)\s*ah\b',0.2,500),
 "voltage_v":   (r'\b(\d+(?:[.,]\d+)?)\s*v\b',0.5,600),
 "power_w":     (r'\b(\d+(?:[.,]\d+)?)\s*w\b',0.1,5000),
 "count":       (r'\b(\d+)\s*(?:pcs|pieces|broches|pin|leds?|canaux|ch)\b',1,10000),
}
def extract_specs(title):
    # IMPORTANT : on garde le point décimal (3.7V) -> surtout PAS norm_raw qui le supprimerait
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii","ignore").decode("ascii").lower().replace("*"," ")
    out={}
    for name,(pat,lo,hi) in SPEC_PATTERNS.items():
        v=[float(x.replace(",",".")) for x in re.findall(pat,t)]; v=[x for x in v if lo<=x<=hi]
        if v: out[name]=max(v)
    return out
UPGRADE_SPECS=["capacity_mah","capacity_ah","power_w","count"]
def primary_spec(specs):
    for k in UPGRADE_SPECS:
        if k in specs: return k,specs[k]
    return None,0.0

# ============================================================================
# BRIQUE 1 — EXTRACTION D'ATTRIBUTS (sur chaque titre, tous les produits)
# ============================================================================
LIION_FF=["18650","21700","26650","14500","16340","18500","20700"]; COIN_FF=["cr2032","cr2016","cr1620","cr2025","cr2450","lr44","ag13"]
def a_form_factor(n):
    for f in LIION_FF:
        if f in n: return f
    for f in COIN_FF:
        if f in n: return "coin"
    if re.search(r'\b(rl20|pile d)\b',n): return "d"
    if re.search(r'\b(rl14|pile c)\b',n): return "c"
    if re.search(r'\b(aaa|lr03|r03)\b',n): return "aaa"
    if re.search(r'\b(aa|lr6|r6)\b',n): return "aa"
    if re.search(r'\b9v\b',n): return "9v"
    if re.search(r'\bnema\s?17\b',n): return "nema17"
    if re.search(r'\bnema\s?23\b',n): return "nema23"
    if "5mm" in n: return "5mm"
    if "3mm" in n: return "3mm"
    for c in ["5050","2835","3528"]:
        if c in n: return c
    return None
def a_chemistry(n,ff):
    if "lipo" in n or "li-po" in n or "polymer" in n: return "lipo"
    if ff in LIION_FF or "li-ion" in n or "liion" in n or "lithium" in n: return "lithium"
    if "nimh" in n or "ni-mh" in n: return "nimh"
    if "alcaline" in n or "alkaline" in n or ff in ("d","c","aa","aaa","9v"): return "alkaline"
    if "plomb" in n or "agm" in n or "ultracell" in n or "acide" in n: return "lead"
    if "vape" in n or "pod" in n: return "vape"
    if ff=="coin": return "lithium"
    return None
def a_connector(n):
    if "type-c" in n or "type c" in n or "usb-c" in n or "usb c" in n: return "usb_c"
    if "micro usb" in n or "micro-usb" in n or "microusb" in n: return "usb_micro"
    if "type-b" in n or "type b" in n or "usb-b" in n: return "usb_b"
    if "xt60" in n: return "xt60"
    if "jst" in n: return "jst"
    if "hdmi" in n: return "hdmi"
    if "rj45" in n or "ethernet" in n: return "rj45"
    if "jack" in n: return "jack"
    if "usb" in n: return "usb_a"
    return None
def a_connectivity(n):
    if "wifi" in n or "wi-fi" in n: return "sans" if ("sans wifi" in n or "without wifi" in n) else "wifi"
    if "bluetooth" in n or re.search(r'\bble\b',n): return "bluetooth"
    if "433" in n or "nrf24" in n or "lora" in n or "zigbee" in n or re.search(r'\brf\b',n): return "rf"
    if "gsm" in n or "gprs" in n: return "gsm"
    if "gps" in n: return "gps"
    if "sans fil" in n: return "wireless"
    return None
def a_control(n):
    if "application" in n or re.search(r'\bappli\b',n) or re.search(r'\bapp\b',n): return "app"
    if "wifi" in n: return "wifi"
    if "bluetooth" in n: return "bluetooth"
    if "telecommand" in n or "radiocommand" in n or re.search(r'\brc\b',n): return "rc"
    return None
def a_vehicle(n):
    w=set(n.split())
    if "voiture" in n or "automobile" in n: return "voiture"
    if "drone" in n or "quadcopter" in n: return "drone"
    if "bateau" in n: return "bateau"
    if "avion" in n: return "avion"
    if "moto" in w: return "moto"
    if "tank" in w: return "char"
    if "robot" in n: return "robot"
    return None
def a_board(n):
    w=set(n.split())
    if "esp32" in n: return "esp32"
    if "esp8266" in n or "nodemcu" in n or "wemos" in n or "esp-12" in n: return "esp8266"
    if "raspberry" in n or "rpi" in w or "pi" in w: return "raspberry"
    if "arduino" in n: return "arduino"
    if w & {"uno","nano","leonardo","mega"}: return "arduino"
    if "microbit" in n: return "microbit"
    if "stm32" in n: return "stm32"
    if "jetson" in n: return "jetson"
    return None
def a_led_form(n):
    if "ruban" in n or "bande" in n or "strip" in n: return "strip"
    if "ampoule" in n or "spot" in n or re.search(r'\be14\b',n) or re.search(r'\be27\b',n) or "gu10" in n: return "bulb"
    if "matrice" in n or "ws2812" in n or "8x8" in n or "neopixel" in n: return "matrix"
    if "cob" in n or "horticole" in n or "spectre" in n: return "cob"
    if "infrarouge" in n or "850nm" in n or "940nm" in n or re.search(r'\bir\b',n): return "ir"
    if "afficheur" in n or "7 segment" in n or "segments" in n: return "display"
    if "neon" in n: return "neon"
    if "5mm" in n or "3mm" in n: return "discrete"
    if "lampe" in n or "projecteur" in n or "luminaire" in n: return "bulb"
    return None
def a_voltage_bucket(specs):
    v=specs.get("voltage_v")
    if v is None: return None
    for b in (3.3,5,12,24,220):
        if abs(v-b)<=max(0.6,b*0.12): return b
    return round(v)
def a_psu(n):
    if "buck" in n or "abaisseur" in n or "step down" in n: return "buck"
    if "boost" in n or "elevateur" in n or "step up" in n: return "boost"
    if "decoupage" in n or "transformateur" in n or "ac dc" in n: return "acdc"
    if "pwm" in n: return "pwm"
    return None
def a_rftech(n):
    if "433" in n: return "433"
    if "bluetooth" in n or re.search(r'\bble\b',n) or "hm-10" in n: return "bluetooth"
    if "wifi" in n: return "wifi"
    if "gsm" in n or "gprs" in n or "sim800" in n: return "gsm"
    if "gps" in n: return "gps"
    if "lora" in n: return "lora"
    if "zigbee" in n: return "zigbee"
    if "nrf24" in n: return "nrf24"
    if "nfc" in n or "rfid" in n: return "nfc"
    return None
def a_component(n):
    for c in ["resistance","condensateur","transistor","zener","diode","quartz","inductance","optocoupleur","thyristor","triac"]:
        if c in n: return c
    return None
def a_module_fn(n):
    if "relais" in n or "relay" in n: return "relais"
    if "lcd" in n or "oled" in n or "afficheur" in n or "ecran" in n: return "display"
    if "l298" in n or "driver" in n: return "driver"
    if "potentiometre" in n or "potentiometer" in n: return "potentiometre"
    if "rtc" in n: return "rtc"
    if "convertisseur" in n: return "convertisseur"
    if "interrupteur" in n: return "interrupteur"
    if "bouton" in n: return "bouton"
    return None
def a_motor(n):
    if "servo" in n or re.search(r'\b(mg90|mg90s|mg995|mg996|sg90|ds3218)\b',n): return "servo"
    if "stepper" in n or "nema" in n or "28byj" in n or "pas a pas" in n: return "stepper"
    if "pompe" in n: return "pompe"
    if "ventilateur" in n: return "ventilateur"
    if "moteur" in n or "motor" in n: return "dc"
    return None
def a_sensor(n):
    if "camera" in n: return "camera"
    if "dht" in n or "ds18b20" in n or "lm35" in n or "temperature" in n or "thermom" in n: return "temperature"
    if "hc-sr04" in n or "ultrason" in n or "sharp" in n or "tof" in n or "distance" in n: return "distance"
    if "pir" in n or "mouvement" in n or "motion" in n or "hc-sr501" in n: return "motion"
    if "mq" in n or "gaz" in n or "co2" in n or "pollution" in n: return "gaz"
    if "bmp" in n or "bme" in n or "pression" in n: return "pression"
    if "acs712" in n or "sct" in n or "courant" in n: return "courant"
    if "mpu" in n or "gyro" in n or "accel" in n: return "imu"
    if "ldr" in n or "lux" in n: return "lumiere"
    if "humidite" in n or "pluie" in n: return "humidite"
    return None

def extract_attrs(title, specs):
    n = norm_raw(title); ff = a_form_factor(n)
    return {"form_factor":ff,"chemistry":a_chemistry(n,ff),"connector":a_connector(n),
            "connectivity":a_connectivity(n),"control":a_control(n),"vehicle":a_vehicle(n),
            "board":a_board(n),"led_form":a_led_form(n),"voltage_bucket":a_voltage_bucket(specs),
            "psu":a_psu(n),"rftech":a_rftech(n),"component":a_component(n),
            "module_fn":a_module_fn(n),"motor":a_motor(n),"sensor":a_sensor(n)}

# ============================================================================
# BRIQUE 2 — Quels attributs définissent « même type » selon la catégorie
# ============================================================================
TYPE_ATTRIBUTES = {
 "batterie":["form_factor","chemistry"], "chargeur":["chemistry","connector"],
 "led":["led_form","voltage_bucket"], "carte":["board","connectivity"],
 "mobilite":["vehicle","control"], "moteur":["motor"], "alimentation":["voltage_bucket","connector"],
 "rf":["rftech"], "composant":["component"], "electronique":["module_fn"],
 "capteur":["sensor"], "connectique":["connector"],
}
# Complémentarité métier (outillage/mecanique/electrique absents => RIEN)
COMPLEMENTARITY_MAP = {
 "batterie":["chargeur","alimentation","composant","connectique"],
 "chargeur":["batterie","connectique"],
 "led":["alimentation","connectique","electronique"],
 "alimentation":["led","electronique","batterie","connectique","carte"],
 "carte":["capteur","connectique","alimentation","led","electronique","moteur","rf"],
 "capteur":["carte","connectique","electronique","alimentation"],
 "moteur":["carte","alimentation","electronique","batterie","mobilite"],
 "mobilite":["batterie","chargeur","moteur","rf","carte"],
 "rf":["carte","alimentation","connectique"],
 "electronique":["carte","connectique","alimentation","composant"],
 "composant":["electronique","connectique","carte"],
 "connectique":["carte","electronique","composant"],
 "solaire":["batterie","alimentation","chargeur"],
 "audio":["carte","alimentation","connectique"],
 "soudure":["composant","connectique"],
}

# ============================================================================
# Construction des colonnes
# ============================================================================
df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["clean_title"] = df["product_title"].apply(normalize_text)
df["category"]    = df["product_title"].apply(infer_category)

# Repli INTELLIGENT : tout produit encore "autre" hérite de la catégorie de son plus proche voisin
_mask = df["category"] == "autre"
if _mask.any():
    _known = df.loc[~_mask]
    _v  = TfidfVectorizer(max_features=3000).fit(df["clean_title"])
    _nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(_v.transform(_known["clean_title"]))
    _, _ind = _nn.kneighbors(_v.transform(df.loc[_mask, "clean_title"]))
    df.loc[_mask, "category"] = _known["category"].to_numpy()[_ind[:, 0]]

df["specs"]   = df["product_title"].apply(extract_specs)
df["attrs"]   = df.apply(lambda r: extract_attrs(r["product_title"], r["specs"]), axis=1)
df["is_tool"] = df["product_title"].apply(is_tool_brand)

uniq = df.drop_duplicates("product_handle")
print("✅ Catégorisation complète + attributs extraits")
dist = uniq["category"].value_counts()
print(f"\n📊 {len(uniq)} produits — distribution des catégories :")
print(dist.to_string())
print(f"\n   couverture : 'autre' = {100*dist.get('autre',0)/len(uniq):.1f}%   "
      f"| produits avec ≥1 attribut = {100*(uniq['attrs'].apply(lambda a: any(v is not None for v in a.values()))).mean():.0f}%")

In [ ]:
# === Affinage : reconnaître la FONCTION d'un circuit intégré (pour les CI nus type 74HC595, NE555…) ===
# Sans ça, un registre à décalage n'avait aucun critère -> recommandations vides.
# Avec ça : 74HC595 ↔ autres registres à décalage ; NE555 ↔ timers ; etc. (précis, pas de mélange)
def a_module_fn(n):
    if "relais" in n or "relay" in n: return "relais"
    if "registre" in n or "decalage" in n or "shift register" in n or "74hc595" in n or "74hc164" in n or "74hc165" in n: return "registre"
    if "rtc" in n or "horloge" in n or "ds1302" in n or "ds3231" in n: return "rtc"
    if "ne555" in n or "timer" in n or re.search(r'\b555\b', n): return "timer"
    if "lcd" in n or "oled" in n or "afficheur" in n or "ecran" in n: return "display"
    if "l298" in n or "uln2003" in n or "driver" in n or "darlington" in n: return "driver"
    if "amplificateur" in n or "opamp" in n or "lm358" in n or "lm741" in n or "tda" in n: return "ampli"
    if "potentiometre" in n or "potentiometer" in n: return "potentiometre"
    if "optocoupleur" in n: return "opto"
    if "convertisseur" in n: return "convertisseur"
    if "interrupteur" in n: return "interrupteur"
    if "bouton" in n: return "bouton"
    if re.search(r'\b74(hc|hct|ls)\d', n): return "logique"   # autre logique 74xx
    return None

# Recalcule les attributs avec la fonction CI améliorée
df["attrs"] = df.apply(lambda r: extract_attrs(r["product_title"], r["specs"]), axis=1)
print("✅ Fonctions des circuits intégrés reconnues (registre / timer / rtc / driver / ampli …)")

## 📈 Étape 3 : Agrégation des produits

In [ ]:
products = df.drop_duplicates(subset=["product_handle"]).copy()

inv = df.groupby("product_handle").agg({
    "available (not editable)":"sum",
    "on hand (current)":"sum",
    "on hand (new)":"sum"
}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]

products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = (
    products["product_title"].fillna("") + " categorie " + products["category"].fillna("")
)
products = products[[
    "product_handle","product_title","category","sku","specs","attrs","is_tool",
    "clean_title","product_profile","available_total","on_hand_current","on_hand_new"
]].reset_index(drop=True)

print(f"✅ {len(products)} produits uniques")
print(f"   En stock: {len(products[products['available_total']>0])}")
products[["product_title","category","attrs","available_total"]].head(5)

## 🤖 Étape 4 : Moteur de recommandation intelligent & adaptatif

Le moteur choisit **les bons critères selon le produit** (table `TYPE_ATTRIBUTES`) :
batterie → *format + chimie* · voiture → *mode de contrôle* · carte → *connectivité* · chargeur → *connecteur*…

Pour tout produit, il renvoie 3 blocs (**STRICT** — rien plutôt qu'un faux) :
- ⬆️ **UPGRADE** — même type, meilleure spec (18650 2600 → 3400 mAh)
- 🔁 **ALTERNATIVE** — même type, équivalent (Arduino *sans wifi* → Arduino, **pas** ESP32)
- 🧩 **GOES-WITH** — complémentaire compatible (Type‑C → Type‑C, ruban 12V → alim 12V, 18650 → chargeur 18650)

In [ ]:
class IoTRecommendationEngine:
    """
    Moteur de recommandation 'intelligent & adaptatif'.

    Pour CHAQUE produit, il choisit les BONS critères selon la catégorie
    (TYPE_ATTRIBUTES) et renvoie 3 buckets — STRICT (rien plutôt qu'un faux) :

      ⬆️ UPGRADE      -> même type, meilleure spec (18650 2600mAh -> 3400mAh)
      🔁 ALTERNATIVE  -> même type, le plus proche (Arduino sans wifi -> Arduino, pas ESP32)
      🧩 GOES-WITH    -> complémentaire compatible (connecteur/tension/chimie : Type-C -> Type-C)

    Si rien de pertinent en stock -> bucket vide (ex : un tournevis n'a pas de complément IoT).
    """
    STRONG = re.compile(r'^(?=.*[a-z])(?=.*\d)[a-z0-9]{3,}$')   # tokens identité : 18650, esp32, 5050...

    def __init__(self, products_df, complementarity_map):
        self.products = products_df.copy().reset_index(drop=True)
        self.comp_map = complementarity_map
        self.products["clean_profile"] = self.products["product_profile"].fillna("").apply(normalize_text)
        self.products["strong"] = self.products["clean_profile"].apply(
            lambda s: {w for w in s.split() if self.STRONG.match(w)})
        self.cat_to_idx = {c: list(g.index) for c, g in self.products.groupby("category")}
        # Colonnes pré-calculées en listes (accès O(1), bien plus rapide que .loc dans les boucles)
        self._cat   = self.products["category"].tolist()
        self._attrs = self.products["attrs"].tolist()
        self._tool  = self.products["is_tool"].tolist()
        self._stock = self.products["available_total"].tolist()
        self._strong= self.products["strong"].tolist()
        self._clean = self.products["clean_profile"].tolist()
        self._specs = self.products["specs"].tolist()

        if HAS_ST:
            print("🔄 Chargement du modèle sémantique...")
            self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
            self.X = self.model.encode([f"passage: {p}" for p in self.products["clean_profile"]],
                                       normalize_embeddings=True, show_progress_bar=False)
            self.is_sparse = False; mode = "sémantique (sentence-transformers)"
        else:
            self.vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
            self.X = self.vec.fit_transform(self.products["clean_profile"])
            self.is_sparse = True; mode = "TF-IDF (repli hors-ligne)"
        self.nn = NearestNeighbors(metric="cosine").fit(self.X)
        print(f"✅ Moteur prêt — {len(self.products)} produits indexés — mode {mode}")

    def get_product_index(self, q):
        if isinstance(q, (int, np.integer)):
            return int(q) if 0 <= q < len(self.products) else None
        m = self.products[self.products["product_title"].str.contains(re.escape(str(q)), case=False, na=False)]
        return int(m.index[0]) if len(m) else None

    def _qvec(self, i): return self.X[i] if self.is_sparse else self.X[i:i+1]

    def _same_type(self, src_cat, src_attrs, src_tool, j):
        """True/False si 'même type', None si pas d'attribut discriminant (=> repli)."""
        if bool(self._tool[j]) != bool(src_tool): return False  # outil ≠ IoT
        req = [a for a in TYPE_ATTRIBUTES.get(src_cat, []) if src_attrs.get(a) is not None]
        if req:
            return all(self._attrs[j].get(a) == src_attrs[a] for a in req)
        return None

    def _rows(self, idx_list, sim):
        cols = ["product_title","category","attrs","specs","available_total"]
        if not idx_list:
            return pd.DataFrame(columns=cols+["similarity"])
        out = self.products.loc[idx_list, cols].copy()
        out["similarity"] = [round(sim.get(j,0.0),3) for j in idx_list]
        return out.reset_index(drop=True)

    def recommend_smart(self, query, n=3, in_stock_only=True, comp_min=0.12):
        i = self.get_product_index(query)
        if i is None: return None
        cat = self._cat[i]; sa = self._attrs[i]; stool = self._tool[i]; src_strong = self._strong[i]
        pk, pv = primary_spec(self._specs[i])
        K = min(len(self.products), 120)
        dist, idx = self.nn.kneighbors(self._qvec(i), n_neighbors=K)
        sim = {int(j): 1-float(d) for d,j in zip(dist[0], idx[0])}

        def ok_stock(j): return (not in_stock_only) or self._stock[j] > 0

        # --- candidats MÊME TYPE (parcourt TOUTE la catégorie, strict adaptatif) ---
        same = []
        for j in self.cat_to_idx.get(cat, []):
            if j == i or not ok_stock(j): continue
            st = self._same_type(cat, sa, stool, j)
            if st is True:
                same.append(j)
            elif st is None:  # pas d'attribut discriminant -> repli : sim forte + token identité partagé
                if bool(self._tool[j])==bool(stool) and sim.get(j,0) >= 0.40 and (src_strong & self._strong[j]):
                    same.append(j)

        # ⬆️ UPGRADE : même type, spec "plus grand=mieux" supérieure
        ups = []
        if pk:
            ups = [j for j in same if primary_spec(self._specs[j])[0]==pk and primary_spec(self._specs[j])[1] > pv]
            ups = sorted(ups, key=lambda j:(primary_spec(self._specs[j])[1], sim.get(j,0)), reverse=True)[:n]
        # 🔁 ALTERNATIVE : même type, le plus proche
        alts = sorted([j for j in same if j not in ups], key=lambda j: sim.get(j,0), reverse=True)[:n]

        # 🧩 GOES-WITH : complémentaire compatible (STRICT, gate de pertinence)
        comp = []
        if not stool:
            comp_cats = self.comp_map.get(cat, [])
            volt_ok = cat not in ("batterie","chargeur")   # tension non pertinente pour une source d'énergie
            scored = []
            for ci, cc in enumerate(comp_cats):
                for j in self.cat_to_idx.get(cc, []):
                    if j == i or not ok_stock(j): continue
                    ja = self._attrs[j]; hard = 0.0
                    if sa.get("connector") and ja.get("connector")==sa["connector"]: hard += 0.5
                    if sa.get("form_factor") and sa["form_factor"] in self._clean[j]: hard += 0.6
                    if sa.get("chemistry") and ja.get("chemistry")==sa["chemistry"]: hard += 0.3
                    if volt_ok and sa.get("voltage_bucket") and ja.get("voltage_bucket")==sa["voltage_bucket"]: hard += 0.4
                    tok = 1 if (src_strong & self._strong[j]) else 0
                    if hard <= 0 and not tok and sim.get(j,0) < 0.25: continue   # gate : signal RÉEL requis
                    score = 0.45*min(hard,1.0) + 0.30*tok + 0.25*sim.get(j,0)
                    if score >= comp_min: scored.append((j, score, ci))
            comp = [j for j,_,_ in sorted(scored, key=lambda x:(-x[1], x[2]))[:n]]

        return {"source_idx":i, "source_title":self.products.at[i,"product_title"], "source_cat":cat,
                "source_attrs":{k:v for k,v in sa.items() if v is not None},
                "primary_spec":(pk,pv), "is_tool":bool(stool),
                "upgrade":self._rows(ups,sim), "alternative":self._rows(alts,sim), "goes_with":self._rows(comp,sim)}


engine = IoTRecommendationEngine(products, COMPLEMENTARITY_MAP)

### 🔑 Configuration du token Hugging Face

Pour éviter les avertissements et bénéficier de taux de requêtes plus élevés, vous pouvez ajouter votre token Hugging Face (HF) aux secrets de Colab. Cliquez sur l'icône "🔑" dans le panneau de gauche, puis ajoutez un nouveau secret nommé `HF_TOKEN` avec la valeur de votre token.

In [ ]:
import os
from pathlib import Path

# Token Hugging Face OPTIONNEL et securise. Le modele est PUBLIC -> token NON requis.
# Jamais de token en dur. Sources : 1) secrets Colab (icone cle)  2) fichier .env local.

def _load_dotenv(path='.env'):
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    _load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    print('Token Hugging Face charge (secret/.env) - invisible dans le code.')
else:
    print('Aucun token HF trouve - pas grave, le modele est public.')


## ⭐ Étape 5 : Test — Recommandations adaptatives

In [ ]:
def _attrs_str(a):
    a = {k:v for k,v in (a or {}).items() if v is not None}
    return ", ".join(f"{k}={v}" for k,v in a.items()) if a else "—"

def show_smart(query, n=3, in_stock_only=True):
    """Affiche les 3 recommandations adaptatives pour un produit sélectionné."""
    res = engine.recommend_smart(query, n=n, in_stock_only=in_stock_only)
    if res is None:
        print(f"❌ Produit introuvable : '{query}'"); return
    print("="*104)
    print(f"🛒 SÉLECTION : {res['source_title'][:88]}")
    flag = "  🔧 outillage (hors IoT)" if res["is_tool"] else ""
    print(f"   catégorie = {res['source_cat']}{flag}   |   attributs = {_attrs_str(res['source_attrs'])}")
    print("="*104)
    for title, dfb in [("⬆️  UPGRADE — même type, meilleure spec", res["upgrade"]),
                       ("🔁 ALTERNATIVE — même type, équivalent",   res["alternative"]),
                       ("🧩 GOES-WITH — complémentaire compatible",  res["goes_with"])]:
        print(f"\n{title}")
        if len(dfb) == 0:
            print("   (aucun équivalent / complément pertinent en stock)"); continue
        for _, r in dfb.iterrows():
            print(f"   • [{r['category']:<12}] {r['product_title'][:50]:<50} "
                  f"sim={r['similarity']:.2f}  ({_attrs_str(r['attrs'])})  stock={r['available_total']}")
    print()

# TEST 1 — Batterie 18650 : doit proposer un 18650 plus gros mAh + un chargeur 18650
show_smart("18650 3.7V 2600MAH")

In [ ]:
# TEST 2 — Cartes : Arduino (sans wifi) ≠ ESP32 (wifi). Doit rester dans la même famille.
show_smart("Arduino UNO")
show_smart("ESP32")

In [ ]:
# TEST 3 — Ruban LED 12V (autres rubans 12V) + alim/Type-C (connecteur identique)
show_smart("Ruban LED RGB 5050")
show_smart("USB TYPE C POUR RASPBERRY")

In [ ]:
# TEST 4 — Voiture robot (similaires + complément) | Tournevis -> RIEN (hors IoT)
show_smart("Kit Robot Voiture")
show_smart("Coffret de tournevis")

# TEST 5 — Circuit intégré nu (cas du coach) : 74HC595 -> autres registres à décalage
#   (in_stock_only=False = on teste la logique même si l'équivalent est hors stock)
show_smart("74HC595N", in_stock_only=False)

## 🔧 Étape 6 : Régler le moteur

Deux leviers simples :

1. **Affiner « quel critère compte par catégorie »** → édite `TYPE_ATTRIBUTES` (Étape 2). Ex : pour `mobilite` on matche sur `vehicle` + `control` (voiture RC ↔ voiture RC).
2. **Affiner la complémentarité** → édite `COMPLEMENTARITY_MAP`. Une catégorie absente (ex : `outillage`) = **aucun complément** (un tournevis n'a pas de complément IoT).

> 📌 **Pour le coach** — pourquoi ce n'est PAS de l'overfitting : aucun modèle entraîné sur nos exemples. Le moteur **lit les attributs de chaque titre** (format, chimie, connectivité, connecteur, mode de contrôle, tension…) et applique la même mécanique à **tous** les 5 774 produits. Sans historique d'achats, des **règles métier expertes** sont le bon choix (recommandeur *à base de connaissances*).

In [ ]:
# --- Exemple : personnaliser la complémentarité, puis recharger le moteur ---
# (catégories valides : batterie, chargeur, led, alimentation, carte, capteur, moteur,
#  mobilite, rf, electronique, composant, connectique, mesure, soudure, solaire, audio)
COMPLEMENTARITY_MAP["led"]      = ["alimentation", "connectique", "electronique"]
COMPLEMENTARITY_MAP["mobilite"] = ["batterie", "chargeur", "moteur", "rf", "carte"]
engine.comp_map = COMPLEMENTARITY_MAP
print("✅ Carte de complémentarité mise à jour!")

# Astuce : tu peux aussi changer les critères de similarité d'une catégorie, ex :
# TYPE_ATTRIBUTES["led"] = ["led_form", "voltage_bucket"]   # puis ré-exécuter l'Étape 4

show_smart("Drone")

In [ ]:
# === BONUS — Découverte AUTOMATIQUE de familles (clustering non supervisé) ===
# Montre que le modèle peut regrouper les produits *tout seul*, sans règles écrites à la main.
from sklearn.cluster import KMeans

K_FAMILIES = 25
Xc = engine.X if not engine.is_sparse else engine.X        # même représentation que le moteur
km = KMeans(n_clusters=K_FAMILIES, random_state=42, n_init=4).fit(Xc)
engine.products["auto_family"] = km.labels_

print(f"🔎 {K_FAMILIES} familles découvertes automatiquement (top mots de chaque famille) :\n")
if engine.is_sparse:
    terms = np.array(engine.vec.get_feature_names_out())
    centers = km.cluster_centers_
    for c in range(K_FAMILIES):
        top = terms[centers[c].argsort()[::-1][:6]]
        n = int((km.labels_ == c).sum())
        print(f"  famille {c:>2} (n={n:>4}) : {', '.join(top)}")
else:
    # En mode sémantique : on résume chaque famille par ses produits les plus fréquents
    for c in range(K_FAMILIES):
        members = engine.products[engine.products["auto_family"] == c]
        sample = ", ".join(members["product_title"].head(3).str[:28])
        print(f"  famille {c:>2} (n={len(members):>4}) : {sample}")

print("\n💡 Lecture pour le coach : le clustering trouve des familles seul, mais reste bruité")
print("   (catalogue large + titres hétérogènes). On garde donc les catégories métier")
print("   pour la production, et le clustering comme preuve de 'self-discovery'.")

## 🔍 Étape 7 : Audit qualité sur TOUS les produits

On lance le moteur sur les 5 774 produits et on vérifie **automatiquement** qu'aucune recommandation n'est illogique (substitut d'une autre catégorie/attribut, complément hors‑carte). Résultat attendu : **0 illogique** (le moteur est strict par construction). Les « blocs vides » sont normaux et voulus (rien plutôt qu'un faux).

In [ ]:
# ============================================================================
# 🔍 AUDIT QUALITÉ — teste TOUS les produits et détecte les recos illogiques
#   in_stock_only=False  ->  on ignore le stock (comme si TOUT était en stock=100),
#   pour évaluer la LOGIQUE pure du modèle, pas la disponibilité.
# ============================================================================
from collections import Counter
import time

P = engine.products
violations = []; n_test = n_sub = n_comp = n_empty = 0
t0 = time.time()
for i in range(len(P)):
    res = engine.recommend_smart(i, n=3, in_stock_only=False)   # <-- stock ignoré
    if res is None: continue
    n_test += 1; sc = res["source_cat"]; sa = res["source_attrs"]
    subs = list(res["upgrade"].itertuples()) + list(res["alternative"].itertuples())
    for r in subs:
        if r.category != sc:
            violations.append(("substitut d'une AUTRE catégorie", res["source_title"], r.product_title))
        for a in [x for x in TYPE_ATTRIBUTES.get(sc, []) if sa.get(x) is not None]:
            if r.attrs.get(a) != sa[a]:
                violations.append((f"substitut attribut '{a}' différent", res["source_title"], r.product_title))
    for r in res["goes_with"].itertuples():
        if r.category not in COMPLEMENTARITY_MAP.get(sc, []):
            violations.append(("complément hors-carte", res["source_title"], r.product_title))
    if subs: n_sub += 1
    if len(res["goes_with"]): n_comp += 1
    if not subs and not len(res["goes_with"]): n_empty += 1

print(f"🔍 Audit LOGIQUE PURE (stock ignoré) de {n_test} produits en {time.time()-t0:.0f}s\n")
print(f"   • avec ≥1 substitut    : {n_sub:>5}  ({100*n_sub/n_test:.0f}%)")
print(f"   • avec ≥1 complément   : {n_comp:>5}  ({100*n_comp/n_test:.0f}%)")
print(f"   • blocs vides (aucun)  : {n_empty:>5}  ({100*n_empty/n_test:.0f}%)  ← produits réellement sans équivalent/complément")
print(f"\n>>> 🚨 Recommandations ILLOGIQUES : {len(violations)}")
if violations:
    for t, c in Counter(v[0] for v in violations).items(): print(f"   - {t} : {c}")
    print("\n   Exemples :")
    for v in violations[:10]: print(f"   • [{v[0]}] {v[1][:36]} ⇒ {v[2][:36]}")
else:
    print("   ✅ AUCUNE — tous les substituts sont du même type, tous les compléments sont cohérents.")